# 〰️ Beyond Linearity
**ISLP Chapter 7 · Pattern Recognition for the Rest of Us**

> Linear regression assumes a straight-line relationship. But many real-world relationships are curved. This notebook covers flexible extensions: polynomial regression, splines, and Generalized Additive Models — all interpretable ways to model nonlinearity.

### What you'll learn
- Polynomial regression: fitting curves with OLS
- Step functions: piecewise constants
- Regression splines: smooth curves with knots
- Natural cubic splines
- GAMs: additive models with one smooth term per feature

### Dataset: Wage (ISLP) + Auto
### Time: ~55 minutes

In [ ]:
# Overview: income vs age relationship
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(wage['age'], wage['wage'], alpha=0.15, color='#888', s=10)
ax.set_xlabel('Age'); ax.set_ylabel('Wage ($)')
ax.set_title('Wage vs Age — clearly nonlinear, needs more than a straight line')
plt.tight_layout(); plt.show()

## 📐 Part 1 — Polynomial Regression

Extend linear regression with polynomial terms:
```
Y = β₀ + β₁X + β₂X² + β₃X³ + ... + βdXd + ε
```
This is still linear regression (linear in the β coefficients) — OLS works perfectly. The trick is just creating the polynomial features from X.

In [ ]:
# Polynomial regression: compare degrees
X_age = wage['age'].values.reshape(-1,1)
y_wage = wage['wage'].values
age_range = np.linspace(wage['age'].min(), wage['age'].max(), 200).reshape(-1,1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
degrees = [1, 2, 4, 10]
for ax, d in zip(axes, degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#1e5fa8', lw=2.5)
    ax.set_title(f'Degree {d}\n10-fold CV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Polynomial Regression: Increasing Degree', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Degree 4 captures the curve. Degree 10 wiggles — overfitting at the extremes')

## 🪢 Part 2 — Splines: Smooth Piecewise Polynomials

Polynomials are global — a wiggle at one end affects predictions everywhere. **Splines** are piecewise polynomials that join smoothly at **knots**.

A **regression spline** with K knots has K+d+1 basis functions (d = degree). The result: flexibility where the data is dense, smoothness everywhere.

**Natural cubic splines** add the constraint that the function is linear beyond the boundary knots — preventing wild behavior at the extremes.

In [ ]:
# Compare: linear, polynomial, spline
from sklearn.preprocessing import SplineTransformer

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_spline = [
    ('Degree-4 Poly', Pipeline([('poly',PolynomialFeatures(4)),('lr',LinearRegression())])),
    ('Spline (4 knots)', Pipeline([('spline',SplineTransformer(n_knots=4, degree=3)),('lr',LinearRegression())])),
    ('Spline (8 knots)', Pipeline([('spline',SplineTransformer(n_knots=8, degree=3)),('lr',LinearRegression())])),
]

for ax, (title, pipe) in zip(axes, models_spline):
    cv_mse = -cross_val_score(pipe, X_age, y_wage, cv=10, scoring='neg_mean_squared_error').mean()
    pipe.fit(X_age, y_wage)
    ax.scatter(wage['age'], wage['wage'], alpha=0.1, color='#888', s=8)
    ax.plot(age_range, pipe.predict(age_range), color='#e85d20', lw=2.5)
    ax.set_title(f'{title}\nCV MSE: {cv_mse:.0f}')
    ax.set_xlabel('Age'); ax.set_ylabel('Wage')
plt.suptitle('Splines vs Polynomial', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## ➕ Part 3 — Generalized Additive Models (GAMs)

GAMs extend linear models to allow nonlinear relationships:
```
Y = β₀ + f₁(X₁) + f₂(X₂) + ... + fₚ(Xₚ) + ε
```
Each fⱼ is a smooth function estimated from the data. The **additive** structure means:
- Each feature contributes independently
- Interpretable: plot each fⱼ to see its effect
- Much more flexible than linear but more interpretable than neural networks

In [ ]:
!pip install pygam -q
from pygam import LinearGAM, s, f, l

# GAM: wage ~ s(age) + s(year) + education
wage_enc = pd.get_dummies(wage[['age','year','education','wage']], drop_first=True, dtype=float)
feature_cols = [c for c in wage_enc.columns if c != 'wage']
X_gam = wage_enc[feature_cols].values
y_gam = wage_enc['wage'].values

gam = LinearGAM(s(0) + s(1) + l(2) + l(3) + l(4) + l(5)).fit(X_gam, y_gam)
print(f'GAM R²: {gam.statistics_["pseudo_r2"]["McFadden"]:.4f}')
gam.summary()

In [ ]:
# Plot GAM smooth terms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, term_idx, label in zip(axes, [0, 1], ['age', 'year']):
    XX = gam.generate_X_grid(term=term_idx)
    pdep, confi = gam.partial_dependence(term=term_idx, width=0.95)
    ax.plot(XX[:,term_idx], pdep, color='#1e5fa8', lw=2.5)
    ax.fill_between(XX[:,term_idx], confi[:,0], confi[:,1], alpha=0.2, color='#1e5fa8')
    ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_xlabel(label); ax.set_ylabel(f'Effect on wage')
    ax.set_title(f'GAM: smooth term for {label}')
plt.suptitle('GAM Partial Effects — Age and Year', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print('📌 Wage rises steeply with age until ~40, then levels off')
print('   Year shows a gradual upward trend — wages rising over time')

In [ ]:
# @title 📝 Quiz — Beyond Linearity
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is a regression spline and how does it differ from a polynomial?
# @markdown **Q2:** What are knots in the context of splines?
# @markdown **Q3:** What is the additive assumption in GAMs?
# @markdown **Q4:** How do you choose the number/location of knots?
# @markdown **Q5:** Name one advantage of splines over high-degree polynomials

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_="Beyond Linearity"
# @title 🤖 AI Grading
# @markdown **Enter your GitHub username, then click ▶ Run.**
# @markdown Colab will send your answers to Gemini automatically — no key needed.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

import textwrap
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {v or '(no answer)'}"
                    for i,(_,v) in enumerate(answers.items(),1))

    prompt = (f"Grade these quiz answers for the \"{_NB_TITLE}\" notebook "
              f"(Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
              f"{qa}\n\n"
              f"For each answer say CORRECT, PARTIAL, or INCORRECT and one sentence why. "
              f"Accept any reasonable paraphrase as correct. "
              f"End with an overall grade (Excellent/Good/Needs Review/Incomplete) "
              f"and one sentence on what to review if they struggled.")

    # Try Colab's built-in Gemini via the generativeai SDK (no key needed in Colab)
    success = False
    try:
        import google.generativeai as genai
        # In Colab, calling generate_content without configure() uses
        # Colab's own managed credentials automatically
        model = genai.GenerativeModel("gemini-1.5-flash")
        resp  = model.generate_content(prompt)
        print("\n" + "\u2550"*56)
        print(f"  \U0001f916 Feedback \u2014 {_NB_TITLE}")
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}")
        print("\u2550"*56)
        for line in resp.text.strip().split("\n"):
            if line.strip():
                for w in textwrap.wrap(line.strip(), 54):
                    print(f"  {w}")
            else:
                print()
        print("\u2550"*56)
        success = True
    except Exception:
        pass

    if not success:
        # Fallback: print the prompt so they can paste it into Colab's AI chat
        print("\u2550"*56)
        print("  Automatic grading unavailable.")
        print("  Paste the text below into the Gemini chat panel:")
        print("  (click the \u2728 sparkle icon in the Colab toolbar)")
        print("\u2550"*56)
        print()
        print(prompt)
        print()
        print("\u2550"*56)

    print("\n  \U0001f4be File \u2192 Save a copy in GitHub \u2192 your fork")
